# HIV Toxicity Classifier — Training on Google Colab

Train the GATv2Conv model on a free T4 GPU (much faster than M-series MPS).

## Quick Steps
1. **Runtime → Change runtime type → GPU** (T4 is free)
2. Run cells top to bottom
3. Upload your project files when prompted
4. Wait ~10-20 min for training to complete
5. Download trained checkpoints

## Project Changelog (every version)

This is the full diary of how we got from a broken prototype to **v5b (shipped)** — plus the v6 (SWA+EMA, wash) and v7 (SMILES augmentation, regressed) attempts that were rolled back. Every change, every blow-up, every fix. Read this before launching a run so you understand what each artifact name means and why the choices are the way they are.

> **Status (April 27, 2026):** Shipped GNN baseline is **v5b** at 0.7739 ± 0.0157 mean test AUC. v6 added SWA+EMA but didn't move the AUC, so it was rolled back to keep the codebase simple. v7 added SMILES augmentation for actives and regressed F0 by 4.4pt, also rolled back. **Current model = v5b.** Next lever is Phase 3 (MolFormer-XL fine-tune) — see end of this notebook. Details in `docs/v5b.md`.

---

### v1 — the inherited starting point (baseline ~0.7788 AUC reported, but inflated)

I inherited a normal-shape GNN setup that was reporting strong AUC numbers, but those values turned out to be heavily inflated — not because the code was buggy in any structural sense, but because it was using **only 3D molecular features** as inputs. With that feature set, the train/val/test splits weren't really being held out the way you'd want for a generalization claim: the 3D-feature-only signal was enough to memorize most of the actives in cluster, and the 0.7788 figure floating around in notes was from that inflated regime. So the model worked, the harness ran end-to-end, the loss went down — it just wasn't measuring what we thought it was measuring.

**What I did:** Rebuilt the feature pipeline so 3D wasn't the only thing the model could see. Split the project into three clean files — `features.py` (atom/bond/scaffold extraction), `model.py` (HIVGNN class), `main.py` (training loop). Switched the GNN backbone to **GATv2Conv** with edge features, residuals, BatchNorm. Added a Murcko-scaffold 5-fold CV via greedy bin-packing of scaffold groups (real held-out evaluation, not the inflated kind). Added focal loss (α=0.75, γ=2.0) instead of vanilla BCE because the imbalance is brutal (**96.5% inactive / 3.5% active**).

**Trouble:** First runs on my Mac's MPS were painfully slow (~3 min/epoch). MPS also flaked on mixed precision in PyTorch 2.9, so I had to gate AMP behind a CUDA check.

---

### v2 — caching and Morgan fingerprints

**What I did:** Added a graph cache (`hiv_preprocessed_cache_v2.pt`) keyed on a SHA-256 of the CSV — rebuild only when the dataset changes. Added Morgan fingerprints (radius 2, 2048 bits) as global features alongside RDKit descriptors, fed through a separate MLP encoder before fusion with the graph readout.

**Result:** Modest improvement, but Morgan FP added ~2k input dims and the gain was small for the parameter cost.

---

### v3 — 3D conformers (the rabbit hole)

**What I did:** Tried adding 3D conformer features — ETKDG embedding, MMFF optimization, distance/angle features. Built `_3d` cache variants.

**Trouble:** 3D embedding fails for ~5–10% of molecules. RDKit's MMFF would silently return None for some, NaN-poison the global feature tensor, and tank training. Also: cache build time went from ~30 sec to ~10 minutes. The signal-to-cost ratio was terrible at this scale.

**Verdict:** Reverted. Kept the scaffolding for later (Phase 4) but removed it from the active path.

---

### v4 / v4_desc — drop Morgan, descriptors only (0.7780 ± 0.0297)

**What I did:** Experiment A — drop Morgan FP entirely, keep only the 54 RDKit descriptors as global features. Toggle via env var `HIV_USE_MORGAN=0` (default). Variant tag in filenames: `_desc` for descriptors-only, `_full` if Morgan is back on.

**Result:** Mean AUC essentially unchanged (0.7780 vs ~0.778 with Morgan), but parameter count dropped sharply and training got faster. **Confirmed: Morgan FP wasn't actually pulling its weight on this benchmark with our model size.**

**Trouble:** Per-fold variance was high — **±0.0297 std**. Some folds at 0.81, others at 0.74. That much fold-to-fold spread on a 5-fold CV usually means the validation signal is noisy or leaky.

---

### v5 — Phase 1.5: kill the validation leakage (0.7667 ± 0.0297)

**The bug:** I had been doing random 90/10 carve-out *within each fold's training scaffolds* to create a val set. That meant val and train shared scaffolds — which on a scaffold-clustered dataset is **leakage**. Val AUC was overfitting to within-scaffold memorization and disagreed with test AUC by up to 10 points.

**What I did:** Nested bin-packing — within each outer fold's training scaffolds, bin-pack again into 20 sub-bins, pool 2 of them as the val set. Now val is **scaffold-held-out from train**, just like test. Also fixed per-fold normalization: descriptor mean/std now fits on the fold's training graphs only (raw-features snapshot pattern, no cross-fold mutation contamination).

**Result on first run:** Mean AUC dropped slightly to **0.7667**, variance still **±0.0297**. The mean drop made sense (we were no longer rewarding leakage), but the variance staying high meant something else was still wrong.

---

### v5b — fix the noisy fold (0.7739 ± 0.0157)

**The bug:** Fold 0 came in at **0.7142** — way below the others. Diagnosis: with `val_pool=2`, fold 0's val set had only 71 positives. That's too few for a stable AUC signal. Val AUC peaked at epoch 1 (0.6703 — basically chance-level on a noisy sample) then drifted down. Early stop fired at epoch 21 because patience kept hitting on tiny noise wiggles.

**What I did:** Two changes —
1. `val_pool=2 → 3` — pulls val_pos up from ~71 to ~126 per fold. Smoother signal.
2. Added `MIN_EPOCHS=30` floor — early stop can't fire before epoch 30 regardless of patience. Protects against early-noise lock-in.

Also added a **per-fold diagnostic block** that prints train/val/test sizes, positive counts, scaffold-leakage check (`train∩val`, `train∩test` — must both be 0), and a warning if val_pos < 30. Lets you spot a degenerate fold immediately instead of after a 20-min training run.

**Result:** Fold 0 recovered to **0.7813** (+6.7 points!). Mean **0.7739**, std **±0.0157** — variance halved. Val and test now track within ~1–2 points instead of 10-point blowouts. Phase 1.5 complete.

---

### v6 — Phase 2: SWA + EMA (rolled back as a wash)

> **Status:** rolled back April 27, 2026. Implementation worked as designed (smoother val curves, clean SWA Option B trigger), but mean test AUC didn't move and per-fold variance widened on some folds. Removed because it added ~150 LOC of training-time complexity for no AUC gain. Kept this entry as design history.

**Goal:** Push from 0.7739 → ~0.79–0.81 with no new data, no architecture changes — just better optimization.

**What I did:**

1. **EMA (Exponential Moving Average)** — per-step shadow copy of model weights, decay=0.999. Validation runs on the EMA weights so the early-stop signal is smoothed (no more 1-batch noise driving patience). The "best" checkpoint we save during training is the best EMA snapshot, not raw.

2. **SWA (Stochastic Weight Averaging) — Option B trigger.** I considered three triggering schemes:
   - **A:** Anchor SWA to a fixed epoch (say 75) → if early stop fires before, SWA collected nothing.
   - **B:** Anchor SWA to early-stop patience — kick in when `patience_counter >= PATIENCE/2` (= 10). That's the "we're plateauing" signal — exactly when averaging helps.
   - **C:** Disable early stopping during SWA → wastes compute on already-converged folds.
   
   I went with **B** because it adapts per fold. Some folds early-stop at epoch 35, others at 80; B guarantees ~10 SWA snapshots either way. SWA also snapshots the **EMA-smoothed** weights, not raw — averaging smoothed snapshots is strictly better than averaging step-noisy ones.

3. **Final 3-way selection.** At the end of each fold I evaluate two candidates on val: best-EMA snapshot, and the SWA average (with `update_bn` pass to recompute BN stats). Whichever wins on val AUC gets saved as the fold checkpoint. The training log prints `[winner=best_ema]` or `[winner=swa]` next to the test AUC so you can see which strategy is winning per fold.

**Cost:** Each epoch is ~5% slower (extra EMA update step + occasional SWA snapshot copy). One extra `update_bn` pass at the end of each fold (~10 sec). Negligible.

**Cache:** Stayed at v5. Graph features are unchanged — SWA/EMA only touches training. Don't want to force a 30-sec cache rebuild for nothing. Filenames bumped: `best_gnn_fold*_v6_desc.pth`, `global_feature_stats_v6_desc*.pt`.

**Expected result:** +0.5 to +2.0 AUC over v5b. SWA+EMA stacking is well-documented on noisy supervised tasks. If it underwhelms, I'll bisect (SWA alone vs EMA alone) before moving to Phase 2.5 (SMILES augmentation).

---

### Persistent cross-cutting troubles & resolutions

- **MPS flakiness on PyTorch 2.9** → Mixed precision and some scatter ops misbehave. Solution: gate AMP behind `DEVICE.type == 'cuda'`. On Mac, just live with the slower MPS run; for real training, use Colab T4.
- **`torch-scatter` / `torch-sparse` install hangs in Colab** → these are C++ wheels that don't always match Colab's preinstalled torch+CUDA, leading to a 10–15 min source build. Solution: skip them. `torch_geometric` is pure Python and runs fine without them. Cost: a harmless `scatter(reduce='max') can be accelerated` warning per batch.
- **Cache invalidation footguns** → fixed via SHA-256 hash of `hiv.csv` + a `cache_version` integer, both stored in the cache file's meta dict. Mismatch on either → rebuild.
- **Cross-fold normalization contamination** → Earlier I was mutating `g.global_features` in-place per fold. That meant fold 4's normalization was sitting on top of folds 0–3's previous mutations. Fix: snapshot raw features once at the top, restore from the snapshot at the start of each fold, then apply fresh normalization.

---

---

### Roadmap — what I might do next, and what it will cost

Cost estimates assume Colab T4 free tier where it works, and rented GPU where it doesn't. **Compute** = wall-clock + electricity. **Cost** = my actual out-of-pocket if I rent.

#### Phase 2.5 — SMILES augmentation (next today, if Phase 2 wins)
**What:** randomize SMILES enumeration per epoch so the GNN sees the same molecule with different atom orderings. Strong data augmentation for graph nets.
**Expected gain:** +1.0 to +2.0 AUC.
**Compute:** +30–50% per epoch (CPU graph rebuild every epoch instead of cached).
**Cost:** **$0** — still fits in Colab T4 free tier.
**Risk:** invasive change to the dataset/cache layer. Easy to revert via git.

### v7 — SMILES augmentation for actives (Phase 2.5): negative result, reverted

**Date:** April 26, 2026 (same day as v6 ship).
**Status:** Aborted mid-run. Code reverted to v6 state. No artifacts shipped.

**What I tried.** Each active molecule expanded into 1 canonical graph + up to 5 atom-permuted variants (`Chem.RenumberAtoms` over a seeded shuffle, deduplicated against the canonical SMILES). Inactives unchanged. Train sees all variants; val/test filtered to canonical-only (otherwise per-molecule confidence inflates into per-variant duplicates and AUC lies). Cache bumped 5 → 7. Artifacts tagged `_v7_`.

**Why I expected it to help.** Permutation-invariant GNNs *should* be invariant to atom order — but in practice node ordering biases message-passing and pooling. Augmenting the minority class with permutation variants was meant to (a) effectively 6× the active gradient signal per epoch, (b) regularize against accidental order-dependence, (c) tighten the val/test gap that v6 still showed on F1.

**What actually happened.** F0 finished at **test=0.7418** vs v6 F0=0.7859 — **4.4pt regression** with a **5.78pt val/test gap** (val 0.7996, test 0.7418). F1 reproduced v6's exact val-grinding pathology — val climbing endlessly past epoch 77 with no plateau — but worse, because the augmentation gave it more permutation variants per active molecule to memorize against. I Ctrl-C'd at that point.

**Why I didn't wait for F2–F4.** "Maybe the later folds will save the mean" is the wrong frame for this project. Two consecutive folds with the same val-overfit pathology is a pattern, not bad luck. A high-variance ensemble (some folds at 0.74, some at 0.80) is worse in deployment than a tight one with slightly lower mean — and I need *all* folds good, not a saving grace.

**Diagnosis.** Three factors, in decreasing order of likely impact:

1. **Augmenting actives doesn't widen scaffold diversity.** Permutation variants share the same Murcko scaffold and canonical SMILES — the model sees more *examples per molecule*, not more *molecules*. The scaffold-held-out test set still contains scaffolds the model has never seen, and augmentation does nothing for those. Meanwhile, on the (in-fold-cluster) val set, the model gets 6× chances to memorize each active. Net effect: stronger val signal, unchanged test signal, wider gap.
2. **Focal loss + augmentation compounded confident-on-wrong-answers.** v6 already used focal (α=0.75, γ=2.0), which heavily weights "hard" examples. Augmentation gave the loss 6× more "active" examples to be confident about — model converged to high-confidence predictions on training-cluster actives that didn't transfer to held-out scaffolds.
3. **SWA was confounded.** SWA fired at epoch 24 on F1 and was winning the val race. We were testing aug+SWA, not pure aug — but even with SWA's regularization, F0 still regressed 4.4pt. So SWA wasn't masking a worse underlying problem.

**Lesson.** Two no-new-data tricks in a row (Phase 2 SWA/EMA, Phase 2.5 augmentation) have now plateaued or regressed. The bottleneck isn't optimization variance or per-molecule example count — it's **scaffold generalization**. Next lever needs to actually expand the model's chemical knowledge, not re-cut the same data. → Phase 3 (ZINC pretraining) becomes the priority instead of "more regularization knobs."

**Code state after revert.** `features.mol_to_randomized_graphs` removed; `main.py` was first restored to v6, then v6 itself was rolled back to v5b on April 27, 2026 (checkpoint paths now `_v5_`). `inference.py` stats path now `_v5_`. Leftover `_v6_` and `_v7_` files on disk are safe to delete.

---

#### Phase 3 — ZINC pretraining (June timeframe)
**What:** pretrain the GNN backbone on ZINC15 (~250M molecules) with self-supervised objectives — masked atom prediction, context prediction, contrastive (GraphCL/MolCLR-style). Then fine-tune on HIV.
**Expected gain:** +2.0 to +4.0 AUC. This is where SOTA-adjacent numbers (0.82+) live.
**Compute:** ~12–24 hrs on a single A100 for a 5–10M parameter backbone. Could go bigger.
**Cost:**
- Conservative (5M params, ZINC15 subset of 10M molecules, 1× A100): **~$30–60** on RunPod / Vast.ai (~$1.50–2/hr × 20 hrs).
- Aggressive (50M params, full ZINC15, 1× A100, 3 days): **~$150**.
- Crazy (750M params, 8× RTX 5090, full ZINC20): **$500+** for one run. Out of scope for a $500 budget.

**Plan:** start at the conservative end. If pretraining lifts test AUC noticeably, scale up.

#### Phase 4 — 3D conformer features (after pretraining)
**What:** revisit what I tried in v3 — but properly this time. Ensemble multiple ETKDG conformers per molecule, MMFF-optimized, distance/angle/torsion features as **edge** attributes (not global). Likely use a 3D-aware backbone like SchNet/DimeNet++ for the conformer stream, fuse with the 2D GATv2Conv stream.
**Expected gain:** +0.5 to +2.0 AUC (smaller than ZINC pretraining for HIV specifically, since the dataset has limited 3D-sensitive activity).
**Compute:** conformer generation is the bottleneck — ~5–15 sec per molecule × 41k molecules × 5 conformers = ~12–30 hrs of CPU. Training is then ~2× v6 wall time.
**Cost:** **~$10–30** if I run conformer generation on a cheap CPU box overnight, $0 if I'm patient on local hardware.

#### Phase 5 — architecture overhaul
**What:** replace the GATv2Conv stack with something stronger. Candidates: **GraphTransformer** (full graph attention with structural encodings), **GINE+JK** with deeper readout, or a **hybrid GNN+Transformer** like Graphormer. Plus stronger global-feature fusion (cross-attention instead of concat).
**Expected gain:** +1.0 to +3.0 AUC if combined with everything above.
**Compute:** Graphormer is ~3–5× our current model size. Each fold ~30–45 min on T4.
**Cost:** **$0–20** — fits on Colab T4 free for a single 5-fold run; rent A100 only if iteration speed matters.

#### Phase 6 — ensembling across architectures
**What:** train v6 (GATv2), Phase 3 (pretrained), Phase 5 (Graphormer) and average probabilities at inference time. Free generalization boost.
**Expected gain:** +0.5 to +1.5 AUC.
**Compute:** none additional beyond training each model.
**Cost:** **$0**.

#### Total budget plan (April → November, $500 ceiling)

| Phase | When | Cost | Cumulative | Realistic AUC ceiling |
|---|---|---|---|---|
| v6 (SWA+EMA) | Today | $0 | $0 | ~0.79 |
| 2.5 (SMILES aug) | Today/tomorrow | $0 | $0 | ~0.80 |
| 3 (ZINC pretrain, conservative) | June | ~$50 | ~$50 | ~0.82 |
| 4 (3D conformers) | August | ~$20 | ~$70 | ~0.83 |
| 5 (Graphormer) | September | ~$20 | ~$90 | ~0.84 |
| 3 (ZINC pretrain, aggressive rerun) | October | ~$150 | ~$240 | ~0.85 |
| 6 (multi-arch ensemble) | November | $0 | ~$240 | ~0.86 |
| **Buffer for failed experiments** | — | ~$260 | $500 | — |

**Honest target:** **0.84–0.86 test AUC** by November on a $500 budget — solidly in "strong open-source baseline" territory. SOTA on HIV (~0.88+ from large pretrained transformers like ChemBERTa-2 or MolFormer-XL) requires 10–100× more compute and is not realistic on this budget. But "second-best practical model someone could actually retrain themselves" is achievable.

**Why SOTA is so expensive:** the SOTA models compound costs across (a) **scale** — 100M–1B+ params, (b) **data** — full ZINC20 + ChEMBL + PubChem, ~1–4B molecules, (c) **multi-task** — pretrain on dozens of properties simultaneously, (d) **3D ensembles** — 5–20 conformers per molecule, (e) **hyperparameter search** — 100s of runs to pick the winner, (f) **ensembling** — 10+ final models. That stack costs **$50k–$500k** on cloud GPUs. We're optimizing for the practical-cost frontier instead.

---


## Step 1 — Verify GPU is Available

If this shows an NVIDIA GPU, you're good. If it errors, go to **Runtime → Change runtime type → GPU**.

In [ ]:
!nvidia-smi

## Step 2 — Install Dependencies

PyTorch is preinstalled in Colab. We just need PyTorch Geometric, RDKit, and a few others.

In [ ]:
# torch_geometric is pure Python and installs in seconds.
# torch-scatter / torch-sparse are optional C++ accelerators; skipping them
# avoids a 10–15 min source build when prebuilt wheels don't match Colab's torch+CUDA.
# Trade-off: a harmless "scatter(reduce='max') can be accelerated" warning per batch.
!pip install -q torch_geometric
!pip install -q rdkit pandas scikit-learn tqdm

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('Device name:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')

## Step 3 — Upload Project Files

Run the cell below. A file picker will appear. **Select all 5 files** at once:
- `features.py`
- `model.py`
- `main.py`
- `inference.py`
- `hiv.csv`

(They're in your `src/` folder on your Mac.)

In [ ]:
from google.colab import files
uploaded = files.upload()
print('\nUploaded files:')
for name in uploaded:
    print(f'  - {name} ({len(uploaded[name])/1024:.1f} KB)')

## Step 4 — Verify Files Are Present

In [ ]:
!ls -la features.py model.py main.py inference.py hiv.csv

## Step 5 — Run Training (v5b: scaffold 5-fold CV with stable val signal)

This is the **shipped GNN baseline** (mean test AUC 0.7739 ± 0.0157). v5b's two stability fixes:
- `val_pool=3` (was 2) — pulls val_pos from ~71 to ~126 per fold, smoother AUC signal
- `MIN_EPOCHS=30` floor — early stop can't fire before epoch 30, prevents noise lock-in

Plus a per-fold diagnostic block that prints sizes, positive counts, and scaffold leakage (`train∩val=0`, `train∩test=0`) before training each fold.

**v6 (SWA+EMA) and v7 (SMILES augmentation) were tried and rolled back.** See `docs/v5b.md` §4 for why.

Default = RDKit descriptors only (54 dims). To toggle Morgan FP back on: `!HIV_USE_MORGAN=1 python main.py`.

Artifacts: `best_gnn_fold{0-4}_v5_desc.pth`, `global_feature_stats_v5_desc*.pt`, `hiv_preprocessed_cache_v5_desc.pt`.

In [ ]:
!python main.py

## Step 6 — Download Trained Models

Saves all 5 fold checkpoints to your local Downloads folder.

In [ ]:
import glob
from google.colab import files

checkpoints = sorted(glob.glob('best_gnn_fold*_v5_desc.pth'))
fold_stats = sorted(glob.glob('global_feature_stats_v5_desc_fold*.pt'))
print(f'Found {len(checkpoints)} checkpoints:')
for ckpt in checkpoints:
    print(f'  - {ckpt}')
print(f'Found {len(fold_stats)} per-fold stats files')

# Download checkpoints + per-fold stats + shared inference stats
for f in checkpoints + fold_stats + ['global_feature_stats_v5_desc.pt']:
    files.download(f)

## Step 7 (Optional) — Download Cache for Future Runs

Saves ~30 sec on subsequent training runs by skipping graph rebuild. Cache is shared between v5 and v6 (graph features unchanged).

In [ ]:
files.download('hiv_preprocessed_cache_v5_desc.pt')

## Step 8 (Optional) — Test Inference Right Here on Colab

Run a quick sanity check with the freshly trained ensemble. **Note:** `inference.py` currently hardcodes the v5 stats filename — update it to v6 locally before this works, or temporarily symlink/rename the stats file on Colab.

In [ ]:
!python inference.py --ensemble-glob 'best_gnn_fold*_v5_desc.pth'

## Done with GNN training!

On your local Mac:
1. Move the downloaded `.pth` files **and `global_feature_stats_v5_desc.pt`** (plus per-fold stats if you want exact training-time normalization) into your project's `src/` folder.
2. Run inference locally:
   ```bash
   .venv/bin/python src/inference.py --ensemble-glob "src/best_gnn_fold*_v5_desc.pth"
   ```

**Total cost: $0**.

Then continue below for **Phase 3 (MolFormer-XL fine-tune)** — the actual lever that should move the AUC needle.

---

# Phase 3 — MolFormer-XL fine-tune (free pretrained backbone)

After v5b shipped at 0.7739 mean test AUC and v6 (SWA+EMA, rolled back as a wash) and v7 (SMILES augmentation, regressed) both failed to move the needle, the next lever is **a pretrained molecular foundation model**. `ibm/MoLFormer-XL-both-10pct` is a 47M-param transformer pretrained by IBM on 1.1B ZINC+PubChem molecules. Fine-tuning it on HIV with the same scaffold 5-fold splits as v5b is the standard Phase-3 playbook — and it costs **$0** because someone else already paid the pretraining bill.

**What this section does:**
1. Install `transformers` + `sentencepiece`
2. Upload `molformer_model.py`, `molformer_train.py`, `ensemble_inference.py`
3. Run 5-fold MolFormer fine-tuning (uses identical scaffold splits as v5b → ensemble-compatible)
4. Download `best_molformer_fold{0..4}.pth`
5. (Locally) ensemble v5b GNN + MolFormer for higher per-fold AUC

## Step 9 — Install MolFormer dependencies

In [ ]:
!pip install -q transformers sentencepiece

## Step 10 — Upload MolFormer source files

Run the cell, then upload these from your local `src/`:
- `molformer_model.py`
- `molformer_train.py`
- `ensemble_inference.py`

(`features.py`, `model.py`, `main.py`, and the v5 cache should already be on Colab from Steps 7–8. The v5 cache is reused for scaffold splits so MolFormer trains on **identical** held-out folds as v5b.)

In [ ]:
from google.colab import files
import os, re

uploaded = files.upload()

def strip_suffix(name):
    base, ext = os.path.splitext(name)
    return re.sub(r' \(\d+\)$', '', base) + ext

for upload_name, content in uploaded.items():
    target = strip_suffix(upload_name)
    if upload_name != target and os.path.exists(upload_name):
        os.remove(upload_name)
    with open(target, 'wb') as f:
        f.write(content)
    print(f'wrote {target} ({len(content)/1024:.1f} KB)')

## Step 11 — Fine-tune MolFormer-XL (5 folds)

Defaults: batch=32, grad_accum=2 (effective 64), 15 epochs, patience=5, focal loss (α=0.75, γ=2.0).

Per-fold wall time on T4: roughly 8–15 min depending on early-stop. Total run: **45–75 min** for all 5 folds.

If you hit OOM, add `--grad-checkpoint` (saves ~30% VRAM, ~25% slower):
```
!python molformer_train.py --grad-checkpoint
```

Smoke-test single fold first:
```
!python molformer_train.py --fold-limit 1 --epochs 3
```

In [ ]:
!python molformer_train.py

## Step 12 — Download MolFormer checkpoints

Each fold checkpoint is ~190 MB (47M params × 4 bytes). Total ~950 MB across 5 folds — bigger than the v6 GNN ensemble but still trivial to download.

In [ ]:
import glob
from google.colab import files

mf_ckpts = sorted(glob.glob('best_molformer_fold*.pth'))
print(f'Found {len(mf_ckpts)} MolFormer checkpoints:')
for ckpt in mf_ckpts:
    size_mb = os.path.getsize(ckpt) / (1024*1024)
    print(f'  - {ckpt} ({size_mb:.0f} MB)')

for ckpt in mf_ckpts:
    files.download(ckpt)

## Step 13 (Optional) — Test the ensemble on Colab

Quick sanity check: average v6 GNN + MolFormer probabilities on a few SMILES.

In [ ]:
!python ensemble_inference.py \
    --gnn-glob 'best_gnn_fold*_v5_desc.pth' \
    --mf-glob 'best_molformer_fold*.pth' \
    --smiles 'CC(=O)OC1=CC=CC=C1C(=O)O' \
    --smiles 'C[C@H](N)C(=O)O' \
    --smiles 'CC1=CN(C(=O)NC1=O)[C@H]2CC[C@H](N3C=NC=N3)O2'

## Done — local next steps

1. Move all 5 `best_molformer_fold*.pth` into your local `src/` folder (alongside the v5b `.pth` files).
2. Run the ensemble locally:
   ```bash
   .venv/bin/python src/ensemble_inference.py \
       --gnn-glob 'src/best_gnn_fold*_v5_desc.pth' \
       --mf-glob 'src/best_molformer_fold*.pth'
   ```
3. Tune `--gnn-weight` if one side is consistently better. Default 0.5 is a flat average; if MolFormer dominates v5b, drop GNN weight to 0.3.

**Realistic expected outcome:**
- MolFormer alone: 0.81–0.83 mean test AUC (vs v5b's 0.7739)
- Ensemble: 0.82–0.84 mean test AUC, with tighter per-fold variance
